# Reference mappings
Review source masters and candidate crosswalks for customers, products, warehouses, and companies. Validate the source ID, source context, and canonical ID before approving a relationship. Run 01 first and clear its integrity exceptions before using suspect data.

In [ ]:
import pandas as pd
from IPython.display import display

from bronze_review import decode_file, inventory, mapping_candidates, open_bronze

bronze = open_bronze()
catalog = inventory(bronze)

latest = {}
manifests = sorted(
    (row for row in catalog if row["kind"] == "batch manifest"),
    key=lambda row: (row["published_at"], row["run_id"]),
)
for manifest in manifests:
    latest[manifest["source"]] = manifest["run_id"]

def review(domain, master_collections):
    masters = [
        row for row in catalog if row["kind"] == "snapshot"
        and row["run_id"] == latest[row["source"]]
        and row["collection"] in master_collections
    ]
    candidates = mapping_candidates(catalog, domain)
    if not masters:
        print("No matching master collections found.")
    if not candidates:
        print("No candidate crosswalk files found.")
    tables = {}
    for row in masters + candidates:
        print(f'\n{row["kind"]} | {row["source"]} | {row["collection"]} | {row.get("status")}')
        print(row["key"])
        if row.get("status") != "ok":
            continue
        records, fields = decode_file(row["key"], bronze.read(row["key"]))
        if records is None:
            print("Open the raw file in its native reader.")
            continue
        table = pd.json_normalize(records, max_level=1) if records else pd.DataFrame(columns=fields)
        if row["kind"] == "inbound file" and "domain" in table.columns:
            table = table.loc[table["domain"].eq(domain)]
            if table.empty:
                print(f"No {domain} rows in this candidate file.")
                continue
        tables[row["key"]] = table
        display(table.head(20))
    return tables

## Customer
Check source customer key, entity scope, and canonical customer ID.

In [ ]:
customer_tables = review("customer", {"Customer", "customers"})

## Product
Check source item code or SKU and canonical product ID. Use UOM as supporting context.

In [ ]:
product_tables = review("product", {"Item", "products", "UOM"})

## Warehouse
Check source location ID, company scope, and canonical warehouse ID.

In [ ]:
warehouse_tables = review("warehouse", {"Warehouse", "warehouses"})

## Company
Check the ERPNext entity and any inbound entity aliases or IDs. Dealer SQL does not capture a company table.

In [ ]:
company_tables = review("company", {"Company"})

Record the selected source object, source key columns, and canonical key column for each approved mapping. Check for blank keys, duplicate scoped keys, conflicting targets, and unmapped source records when the crosswalk is confirmed.